# Uzbek translation audio — timings on Colab

Builds the per-ayah timings for the newbooks.uz Uzbek translation audio on a free Colab GPU instead
of the Mac (`scripts/uz_audio/README.md`, `docs/MULTILINGUAL_PLAN.md` §8.1).

**What comes out (section 8 downloads it as one zip):**

| File | What it is |
|---|---|
| `data/uz_audio_timings.json` | frame-exact start/end of all 6236 ayahs, read by the bot and the Mini App |
| `data/uz_audio_manifest.csv` | one row per ayah: start/end seconds, frames, **exact byte range** and URL, confidence, what the narrator said (Whisper), reference text. Opens in Google Sheets |
| `cache/` | transcripts, pause profiles and alignments, for `scripts.uz_audio.debug` on the Mac |

**How it works:** it clones the repo (branch `feat/uz-translation-audio`) and downloads the 114 mp3s to *Colab's
temporary disk* (never Drive). It transcribes each surah with Whisper (word timestamps) and records
its pauses, then runs the repo's own `align` and `export`. Work is saved to Google Drive
(`MyDrive/quranayat_uz_audio/`) after every surah, so after a disconnect *Run all* continues where it
stopped.

**Start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. All 33.8 h of audio
takes roughly 3–5 h with large-v3 (1.5–3 h with turbo).

**Then, on the Mac** (seconds, low memory), from the repo root:
```bash
unzip -o ~/Downloads/uz_audio_colab.zip -d /tmp/uz_audio_colab
cp /tmp/uz_audio_colab/data/uz_audio_* data/
mkdir -p ~/.cache/quranayat/uz_audio && cp -R /tmp/uz_audio_colab/cache/* ~/.cache/quranayat/uz_audio/
venv/bin/python -m pytest tests/test_uz_audio_timings.py
```

## 1. GPU check

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
assert gpu.returncode == 0, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
print(gpu.stdout.strip())

## 2. Google Drive for results (a disconnect loses nothing)

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/quranayat_uz_audio"  # transcripts, pauses, alignments: no audio
MP3 = "/content/mp3"  # Colab's temporary disk, never Drive
for d in (f"{OUT}/transcripts", f"{OUT}/energy", f"{OUT}/align", MP3):
    os.makedirs(d, exist_ok=True)
os.environ["UZ_AUDIO_CACHE"] = OUT  # read by scripts.uz_audio.common at import
done = sorted(f for f in os.listdir(f"{OUT}/transcripts") if f.endswith(".json"))
print(f"{len(done)} of 114 surahs already transcribed")

## 3. Install and get the code

In [ ]:
!pip -q install "faster-whisper>=1.1.1,<2"
!rm -rf /content/repo && git clone -q --depth 1 --branch feat/uz-translation-audio https://github.com/otabek-rizakulov-cntxt/QuranAyat_bot.git /content/repo

In [ ]:
import sys

sys.path.insert(0, "/content/repo")
from scripts.uz_audio import common  # noqa: E402

common.MP3_DIR = MP3  # audio stays on Colab's disk
print("code at", common.REPO, "| cache at", common.CACHE)

## 4. Pick the model (about 5 min, optional but recommended)

There is no "Whisper v4": OpenAI's newest are **large-v3** and **large-v3-turbo** (large-v3 with 4
decoder layers instead of 32, which makes it faster but usually a bit less accurate on Uzbek). There is
also a community **Uzbek fine-tune** of turbo. For our cuts, what matters is *where segments start*,
more than spelling.

This cell fetches only the 104 s of al-Baqarah around the three cuts approved by ear (2:284 starts at
9034.1 s, 2:285 at 9067.4 s, 2:286 at 9114.2 s), using one ~1.2 MB Range request. It runs each
candidate and prints its text plus how far its nearest segment start lands from each true cut.
Set `MODEL` in section 6 to the winner: closest starts first, readable text second.

In [ ]:
import time

import numpy as np
import requests
from faster_whisper import WhisperModel

A, B = 9018, 9122  # seconds; al-Baqarah is 96 kbps -> 12 000 bytes/s, 432-byte frames
lo, hi = A * 12000 // 432 * 432, B * 12000 // 432 * 432
r = requests.get(
    "https://newbooks.uz/audio/t1/002.mp3",
    timeout=120,
    headers={"User-Agent": "Mozilla/5.0", "Range": f"bytes={lo}-{hi - 1}"},
)
with open("/content/window.mp3", "wb") as f:
    f.write(r.content)
offset = lo / 12000
# our own decoder: faster-whisper's decode_audio breaks on the newer PyAV Colab ships
window = common.to_float(common.decode_16k("/content/window.mp3"))
TRUE_CUTS = [9034.1, 9067.4, 9114.2]

CANDIDATES = ["large-v3", "large-v3-turbo", "hostmepanda/whisper-large-v3-turbo-uzbek-ct2"]
for name in CANDIDATES:
    try:
        m = WhisperModel(name, device="cuda", compute_type="float16")
    except Exception as err:
        print(f"{name}: could not load ({err})")
        continue
    t0 = time.time()
    segs = list(
        m.transcribe(
            window,
            language="uz",
            word_timestamps=True,
            condition_on_previous_text=False,
            vad_filter=False,
        )[0]
    )
    starts = [offset + s.start for s in segs]
    errs = [min(abs(st - c) for st in starts) for c in TRUE_CUTS]
    print(
        f"\n=== {name}   {time.time() - t0:.0f}s   cut errors: "
        + ", ".join(f"{e:.2f}s" for e in errs)
        + f"   (mean {np.mean(errs):.2f}s)"
    )
    for s in segs:
        print(f"  {offset + s.start:8.2f}  {s.text.strip()}")
    del m

## 5. Download the mp3s (resumable, checked against the server's size)

In [ ]:
import concurrent.futures as cf
import time

import requests

SOURCE = "https://newbooks.uz/audio/t1/{:03d}.mp3"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def fetch(n):
    url, path = SOURCE.format(n), f"{MP3}/{n:03d}.mp3"
    want = int(requests.head(url, headers=HEADERS, timeout=60).headers["Content-Length"])
    for attempt in range(8):
        have = os.path.getsize(path) if os.path.exists(path) else 0
        if have == want:
            return want
        try:
            h = dict(HEADERS, Range=f"bytes={have}-") if have else HEADERS
            with requests.get(url, headers=h, stream=True, timeout=120) as resp:
                resp.raise_for_status()
                with open(path, "ab" if have else "wb") as out:
                    for chunk in resp.iter_content(1 << 20):
                        out.write(chunk)
        except requests.RequestException:
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"surah {n}: download incomplete")


todo = [n for n in range(1, 115) if not os.path.exists(f"{OUT}/transcripts/{n:03d}.json")]
started = time.time()
with cf.ThreadPoolExecutor(12) as pool:  # newbooks.uz gives ~85 KB/s per connection
    total = sum(pool.map(fetch, todo))
print(f"{len(todo)} files, {total / 1e9:.2f} GB in {(time.time() - started) / 60:.1f} min")

## 6. Transcribe

In [ ]:
import dataclasses
import json

import numpy as np
from faster_whisper import WhisperModel

# Set from section 4's comparison. large-v3 (full) is the default: on a GPU the extra
# accuracy is worth ~2-3x the time (roughly 3-5 h on a T4 for all 33.8 h; resumable).
MODEL = "large-v3"
HOP = 320  # 20 ms at 16 kHz, as scripts/uz_audio/transcribe.py
model = WhisperModel(MODEL, device="cuda", compute_type="float16")


def energy_db(x):
    n = len(x) // HOP
    frames = x[: n * HOP].reshape(n, HOP)
    return (20 * np.log10(np.sqrt((frames**2).mean(axis=1)) + 1e-9)).astype(np.float32)


def transcribe(n):
    path = common.mp3_path(n)
    info = common.probe(path)  # proves constant bitrate; its facts go into the transcript
    audio = common.to_float(common.decode_16k(path))  # not decode_audio: see section 4
    segments, _ = model.transcribe(
        audio,
        language="uz",
        word_timestamps=True,
        condition_on_previous_text=False,  # one bad segment must not snowball
        vad_filter=False,  # pauses are our signal; keep every one
    )
    out = [
        {
            "start": round(s.start, 2),
            "end": round(s.end, 2),
            "text": s.text.strip(),
            "words": [
                [round(w.start, 2), round(w.end, 2), w.word.strip()] for w in (s.words or [])
            ],
        }
        for s in segments
    ]
    np.save(f"{OUT}/energy/{n:03d}.npy", energy_db(audio))
    doc = {
        "surah": n,
        "model": "faster-whisper/" + MODEL,
        "duration": info.duration,
        "mp3": dataclasses.asdict(info),
        "segments": out,
    }
    tmp = f"{OUT}/transcripts/{n:03d}.json.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(doc, f, ensure_ascii=False)
    os.replace(tmp, f"{OUT}/transcripts/{n:03d}.json")  # only a finished surah counts as done
    os.remove(path)  # free Colab's disk as we go
    return info.duration, len(out)


audio_done, started = 0.0, time.time()
for n in todo:
    t0 = time.time()
    secs, segs = transcribe(n)
    audio_done += secs
    speed = audio_done / (time.time() - started)
    print(
        f"{n:03d}  {secs / 60:6.1f} min audio  {time.time() - t0:5.0f}s  {segs:4d} segments"
        f"  ({speed:.1f}x real time)",
        flush=True,
    )
print("all transcribed")

## 7. Align and export (CPU, a minute or two)

The repo's own `align` and `export`: ayah boundaries from the transcripts and pauses, then the
timing table and the per-ayah manifest. The `!` lines are problems worth a listen; *low-confidence*
counts are normal for an unverified surah.

In [ ]:
missing = [n for n in range(1, 115) if not os.path.exists(f"{OUT}/transcripts/{n:03d}.json")]
assert not missing, f"not transcribed yet: {missing} - run sections 5 and 6 again"

In [ ]:
!cd /content/repo && python -m scripts.uz_audio.align && python -m scripts.uz_audio.export

## 8. Download the results

In [ ]:
import csv
import shutil

from google.colab import files

stage = "/content/uz_audio_colab"
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(f"{stage}/data")
for name in ("uz_audio_timings.json", "uz_audio_manifest.csv"):
    shutil.copy(f"/content/repo/data/{name}", f"{stage}/data/{name}")
shutil.copytree(OUT, f"{stage}/cache")
zip_path = shutil.make_archive(stage, "zip", stage)
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")

with open(f"{stage}/data/uz_audio_manifest.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "ayahs; for example:")
for row in rows:
    if (row["surah"], row["ayah"]) in {("2", "284"), ("2", "285")}:
        print(
            " ",
            {k: row[k] for k in ("surah", "ayah", "start_s", "end_s", "byte_start", "byte_end")},
        )

files.download(zip_path)